# Bài 58: Độ sâu ảnh nổi với các mô hình nền tảng (Stereo Depth with Foundation Models)

Bài 22 đã xây dựng so khớp ảnh nổi cổ điển: trượt một cửa sổ nhỏ dọc theo các đường quét tương ứng, so sánh các mảng cường độ, lấy độ chênh lệch có chi phí thấp nhất. Nó hoạt động tốt trên các bề mặt có kết cấu và thất bại một cách dễ đoán trên các vùng phẳng, ít kết cấu — **bài toán khẩu độ**, nơi một cửa sổ nhỏ thực sự không thể biết độ dịch nào là đúng. Bài này xây dựng một mạng ảnh nổi *học được* nhỏ (một khối chi phí (cost volume) khả vi cộng với một bước tinh chỉnh tích chập, ý tưởng cốt lõi đằng sau các hệ thống ảnh nổi sâu hiện đại cho đến **FoundationStereo**, <a href="../references.html#wen-2024-foundationstereo">Wen et al., 2025</a>) và cho thấy cụ thể tại sao việc học thắng so khớp cục bộ thuần túy: trường tiếp nhận của một mạng cho phép nó vay mượn ngữ cảnh từ *bên ngoài* vùng mơ hồ, điều mà một cửa sổ cục bộ cố định về mặt cấu trúc không thể làm.

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Một cặp ảnh nổi với một dải không kết cấu có chủ đích (A stereo pair with a deliberately textureless band)
Tạo một dải "thế giới" có kết cấu, đủ rộng để việc dịch nó tạo ra một cặp trái/phải hợp lệ ở một độ chênh lệch nguyên đã biết — thiết lập ảnh nổi tổng hợp ở Bài 22, mở rộng thêm một điểm xoắn: một dải ngang được làm phẳng thành cường độ hằng số, hoàn toàn không có kết cấu bên trong.

In [ ]:
SIZE = 32
MAX_DISP = 8

def make_stereo_pair(rng, size=SIZE, max_disp=MAX_DISP, textureless=True):
    world = rng.uniform(0, 1, (size, size + max_disp)).astype(np.float32)
    world = cv2.GaussianBlur(world, (3, 3), 0)
    for _ in range(15):  # a few sharp features, so texture is patchy rather than uniform
        cx, cy = rng.integers(0, size + max_disp), rng.integers(0, size)
        world[max(0, cy-1):cy+2, max(0, cx-1):cx+2] = rng.uniform(0, 1)

    disparity = rng.integers(1, max_disp)
    band = None
    if textureless:
        b0 = rng.integers(0, size - size // 3)
        band = slice(b0, b0 + size // 3)
        world[band, :] = 0.5  # a flat strip: no texture, so no local cue to match against

    left = world[:, max_disp:max_disp + size]
    right = world[:, max_disp - disparity: max_disp - disparity + size]
    disp_map = np.full((size, size), float(disparity), dtype=np.float32)
    return left.astype(np.float32), right.astype(np.float32), disp_map, band

rng = np.random.default_rng(11)
left0, right0, disp0, band0 = make_stereo_pair(rng)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(left0, cmap='gray'); axes[0].set_title('left image'); axes[0].axis('off')
axes[1].imshow(right0, cmap='gray'); axes[1].set_title('right image'); axes[1].axis('off')
axes[2].imshow(disp0, cmap='viridis'); axes[2].set_title(f'true disparity ({disp0[0,0]:.0f} px)'); axes[2].axis('off')
for ax in axes:
    ax.axhline(band0.start, color='red', linestyle='--', linewidth=0.7)
    ax.axhline(band0.stop, color='red', linestyle='--', linewidth=0.7)
plt.show()

## So khớp khối cổ điển, và nơi nó thất bại (Classical block matching, and where it fails)
So khớp khối ở Bài 22: với mỗi điểm ảnh, so sánh một cửa sổ nhỏ với các cửa sổ ứng viên ở mọi độ chênh lệch, giữ lại cặp khớp có chi phí thấp nhất.

In [ ]:
def block_match(left, right, max_disp=MAX_DISP, win=3):
    H, W = left.shape
    half = win // 2
    disp_map = np.zeros((H, W), dtype=np.float32)
    left_pad = np.pad(left, half, mode='edge')
    right_pad = np.pad(right, half, mode='edge')
    for y in range(H):
        for x in range(W):
            patch_l = left_pad[y:y+win, x:x+win]
            best_d, best_cost = 0, np.inf
            for d in range(max_disp):
                x2 = x + d
                if x2 + win > right_pad.shape[1]:
                    continue
                patch_r = right_pad[y:y+win, x2:x2+win]
                cost = np.sum((patch_l - patch_r) ** 2)
                if cost < best_cost:
                    best_cost, best_d = cost, d
            disp_map[y, x] = best_d
    return disp_map

bm_disp0 = block_match(left0, right0)
err_textured = np.abs(bm_disp0[:band0.start, :] - disp0[:band0.start, :]).mean()
err_textureless = np.abs(bm_disp0[band0, :] - disp0[band0, :]).mean()
print(f'block matching MAE, textured region:    {err_textured:.2f} px')
print(f'block matching MAE, textureless region: {err_textureless:.2f} px  (true disparity = {disp0[0,0]:.0f})')

So khớp khối gần như chính xác ở nơi có kết cấu, và sai nghiêm trọng bên trong dải phẳng — một cửa sổ nhỏ ở đó thực sự không chứa thông tin nào để phân định độ chênh lệch, nên nó khóa vào bất cứ nhiễu nào tình cờ trông tốt hơn một chút. Không có cách phân ngưỡng thông minh nào khắc phục được điều này: cửa sổ quá nhỏ để thấy bất cứ thứ gì khác.

## Một khối chi phí học được (A learned cost volume)
Các mạng ảnh nổi sâu (PSMNet, RAFT-Stereo, và giờ là FoundationStereo) thay phép so sánh cửa sổ chọn thủ công bằng ba thành phần học được:

1. Một CNN nhỏ trích xuất một **bản đồ đặc trưng** từ mỗi ảnh (không phải điểm ảnh thô — các đặc trưng học được, vững hơn so với chỉ cường độ).
2. Một **khối chi phí (cost volume)** được xây dựng bằng cách tương quan bản đồ đặc trưng trái với bản đồ đặc trưng phải, được dịch theo mọi độ chênh lệch ứng viên — cùng ý tưởng mọi-độ-chênh-lệch-cùng-lúc như phép tìm kiếm của so khớp khối, chỉ là thực hiện ở cấp đặc trưng.
3. Một **mạng tinh chỉnh (refinement network)** (thêm vài lớp tích chập, làm việc trên toàn bộ khối chi phí) chuyển một phân phối xác suất mềm, mượt trên các độ chênh lệch thành một ước lượng cuối cùng — bước này có trường tiếp nhận trải rộng trên phần ảnh lớn hơn nhiều so với bất kỳ cửa sổ cục bộ nào, nên nó có thể vay mượn ngữ cảnh từ xa bên ngoài một vùng mơ hồ.

In [ ]:
class FeatureNet(nn.Module):
    def __init__(self, out_ch=8):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, out_ch, 3, padding=1))

    def forward(self, x):
        return self.net(x)

class StereoNet(nn.Module):
    def __init__(self, max_disp=MAX_DISP):
        super().__init__()
        self.feat = FeatureNet()
        self.max_disp = max_disp
        self.refine = nn.Sequential(
            nn.Conv2d(max_disp, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, max_disp, 3, padding=1),
        )

    def forward(self, left, right):
        fl, fr = self.feat(left), self.feat(right)
        cost_volume = []
        for d in range(self.max_disp):
            shifted = fr if d == 0 else F.pad(fr[:, :, :, :-d], (d, 0))
            cost_volume.append((fl * shifted).sum(dim=1))  # correlation at this disparity
        cost_volume = self.refine(torch.stack(cost_volume, dim=1))  # regularize across space
        probs = F.softmax(cost_volume, dim=1)
        disp_range = torch.arange(self.max_disp, dtype=torch.float32).view(1, -1, 1, 1)
        return (probs * disp_range).sum(dim=1)  # soft-argmin: expected disparity, sub-pixel

rng2 = np.random.default_rng(11)
N = 300
lefts, rights, disp_maps = [], [], []
for _ in range(N):
    l, r, d, _ = make_stereo_pair(rng2)
    lefts.append(l); rights.append(r); disp_maps.append(d)
lefts, rights, disp_maps = np.array(lefts), np.array(rights), np.array(disp_maps)

split = int(0.85 * N)
Ltr, Rtr, Dtr = lefts[:split], rights[:split], disp_maps[:split]
Lte, Rte, Dte = lefts[split:], rights[split:], disp_maps[split:]

torch.manual_seed(0)
model = StereoNet()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Lt = torch.tensor(Ltr).unsqueeze(1); Rt = torch.tensor(Rtr).unsqueeze(1); Dt = torch.tensor(Dtr)
for _ in range(300):
    opt.zero_grad()
    loss = F.l1_loss(model(Lt, Rt), Dt)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_te = model(torch.tensor(Lte).unsqueeze(1), torch.tensor(Rte).unsqueeze(1))
mae = (pred_te - torch.tensor(Dte)).abs().mean().item()
print(f'learned stereo network MAE (overall, includes textureless bands): {mae:.3f} px')

## Đối đầu trực tiếp, cụ thể trên dải không kết cấu (Head to head, on the textureless band specifically)
So sánh cả hai phương pháp trên một cặp mới duy nhất, giới hạn chính xác ở các điểm ảnh bên trong dải phẳng, không đặc trưng — chính là nơi cửa sổ cục bộ của so khớp khối không có gì để dựa vào.

In [ ]:
test_rng = np.random.default_rng(99)
left1, right1, disp1, band1 = make_stereo_pair(test_rng)
bm_disp1 = block_match(left1, right1)
with torch.no_grad():
    learned_disp1 = model(torch.tensor(left1[None, None]), torch.tensor(right1[None, None]))[0].numpy()

bm_err_band = np.abs(bm_disp1[band1, :] - disp1[band1, :]).mean()
learned_err_band = np.abs(learned_disp1[band1, :] - disp1[band1, :]).mean()
print(f'textureless-band MAE: block matching = {bm_err_band:.2f} px, learned = {learned_err_band:.2f} px')

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(left1, cmap='gray'); axes[0].set_title('left'); axes[0].axis('off')
axes[1].imshow(disp1, cmap='viridis', vmin=0, vmax=MAX_DISP); axes[1].set_title('true disparity'); axes[1].axis('off')
axes[2].imshow(bm_disp1, cmap='viridis', vmin=0, vmax=MAX_DISP); axes[2].set_title('block matching'); axes[2].axis('off')
axes[3].imshow(learned_disp1, cmap='viridis', vmin=0, vmax=MAX_DISP); axes[3].set_title('learned'); axes[3].axis('off')
for ax in axes:
    ax.axhline(band1.start, color='red', linestyle='--', linewidth=0.7)
    ax.axhline(band1.stop, color='red', linestyle='--', linewidth=0.7)
plt.show()

Sai số của mạng học được bên trong dải phẳng thấp hơn nhiều so với của so khớp khối. Không có gì trong chính dải đó trở nên ít mơ hồ hơn — các phép tích chập của mạng tinh chỉnh có trường tiếp nhận vươn xa ra ngoài các điểm ảnh của chính dải, nên ước lượng độ chênh lệch cho một điểm ảnh ở giữa vùng phẳng bị ảnh hưởng bởi bằng chứng từ các điểm ảnh có kết cấu xung quanh và phía trên nó. Một cửa sổ 3x3 cố định không bao giờ làm được điều này; nó chỉ luôn thấy những gì nằm ngay bên trong nó.

## FoundationStereo, và "nền tảng" bổ sung gì ở đây (FoundationStereo, and what "foundation" adds here)
**FoundationStereo** (<a href="../references.html#wen-2024-foundationstereo">NVIDIA, 2025</a>) chính là công thức này — trích xuất đặc trưng, khối chi phí, tinh chỉnh học được (qua một lược đồ cập nhật lặp suy ra từ RAFT và RAFT-Stereo, thay vì softmax một lần dùng ở đây) — được huấn luyện ở quy mô mô hình nền tảng trên hỗn hợp các tập dữ liệu ảnh nổi tổng hợp lớn và dữ liệu thu thập thực tế, đặc biệt để tổng quát hóa sang các camera, cảnh và miền *mới* mà không cần tinh chỉnh cho từng tập dữ liệu. Đó là cùng câu chuyện "học một tiên nghiệm ở quy mô lớn, dùng nó zero-shot" như CLIP ở Bài 52 và SAM ở Bài 54, giờ áp dụng cho độ sâu theo đơn vị thực từ một cặp ảnh nổi đã hiệu chỉnh thay vì một ảnh đơn mắt duy nhất — điều này né tránh hoàn toàn bài toán mơ hồ tỉ lệ cơ bản ở Bài 56, vì một đường cơ sở ảnh nổi đã hiệu chỉnh mang lại cho độ chênh lệch một ý nghĩa hình học thực sự (Bài 22-24) mà một ảnh đơn lẻ không bao giờ có.

### Bài tập (Exercises)
1. Tăng độ rộng của dải không kết cấu từ `size // 3` lên `size // 2` trong `make_stereo_pair`. Lợi thế của mạng học được so với so khớp khối tăng lên, thu hẹp, hay giữ nguyên khi vùng mơ hồ lớn hơn so với ảnh?
2. Bước `refine` của `StereoNet` gồm hai lớp tích chập với kernel 3x3 mỗi lớp, cho một trường tiếp nhận hạn chế. Thêm một lớp tích chập thứ ba và chạy lại phép so sánh dải không kết cấu — một trường tiếp nhận lớn hơn (với tới nhiều ngữ cảnh xung quanh hơn) có cải thiện thêm ước lượng độ chênh lệch của dải không?
3. Loại bỏ hoàn toàn mạng `refine` (dùng trực tiếp `probs = F.softmax(cost_volume, dim=1)` trên khối chi phí tương quan thô, không có bước chính quy hóa học được nào) và huấn luyện lại. Hiệu năng trên dải không kết cấu của mạng có sụp đổ trở lại gần với so khớp khối không, xác nhận rằng ngữ cảnh không gian — không phải chỉ riêng các đặc trưng học được — là thứ thu hẹp khoảng cách?